# PS5 Pro Search Agent — APK-Build in Google Colab

Dieses Notebook baut aus `main.py`, `buildozer.spec` und `icon.png` eine installierbare
Android-APK (Kivy + python-for-android, arm64-v8a und armeabi-v7a).

**Ablauf**

1. Build-Umgebung installieren (~3 Min.)
2. Projekt-ZIP hochladen (oder aus Google Drive einbinden)
3. `buildozer android debug` ausführen (10–40 Min., je nach Cache)
4. APK herunterladen und auf dem Handy installieren

**Vorher:** `main.py`, `buildozer.spec` und `icon.png` in einen Ordner legen und den Ordner
als ZIP packen (z. B. `ps5pro.zip`). Das ZIP darf den Ordner enthalten oder die Dateien
direkt — beides wird erkannt.

**Wichtig:** Colab beendet Sitzungen (Inaktivität, Verbindungsabbruch oder 12-Stunden-Limit).
Nutze die optionale Cache-Zelle, damit ein zweiter Lauf nicht wieder bei Null beginnt.

## 1) Systempakete installieren

Java 17 (Gradle/AGP), die Autotools für die Native-Builds von Python und Kivy sowie
`python3.10-venv`. Die Ausgabe ist absichtlich knapp gehalten — `java -version` am Ende
zeigt, dass die Installation geklappt hat.

In [ ]:
%%bash
set -uo pipefail
apt-get -qq update
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y \
  autoconf automake libtool pkg-config zlib1g-dev libncurses-dev libffi-dev \
  libssl-dev cmake unzip zip ccache git libltdl-dev > /dev/null
# Java 17 ist der Ziel-JDK; auf neueren Images gibt es nur noch 21.
for JDK in openjdk-17-jdk-headless openjdk-21-jdk-headless; do
  if DEBIAN_FRONTEND=noninteractive apt-get -qq install -y "$JDK" > /dev/null 2>&1; then
    echo "JDK installiert: $JDK"; break
  fi
done
ls -d /usr/lib/jvm/* 2>/dev/null | head -5
java -version 2>&1 | head -2 || echo "kein java gefunden - Zelle pruefen"


## 2) Buildozer in eigener Python-3.11.9-Umgebung installieren

python-for-android ist auf sehr neuen Python-Versionen empfindlich, deshalb legt `uv`
eine virtuelle Umgebung mit **Python 3.11.9** an - unabhängig davon, welches Python die
Colab-Laufzeit gerade mitbringt (3.11/3.12/3.13). `cython==0.29.37` ist die
Version, mit der der Kivy-2.3.0-Recipe sauber baut. Buildozer kommt aus dem Git-Master,
weil die PyPI-Version 1.5.0 von 2024 nicht mehr zum aktuellen python-for-android passt.

In [ ]:
%%bash
set -uo pipefail
# Host-Python muss <= 3.12 sein: dafuer ist python-for-android "master" freigegeben.
# Colab liefert je nach Laufzeit Python 3.11, 3.12 oder 3.13 - "uv" pinnt 3.12 fest.
command -v uv >/dev/null 2>&1 || curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
uv python install 3.12
rm -rf /content/bzenv
uv venv --python 3.11.9 /content/bzenv
/content/bzenv/bin/python -V
uv pip install --python /content/bzenv/bin/python --upgrade pip setuptools wheel
uv pip install --python /content/bzenv/bin/python \
  "git+https://github.com/kivy/buildozer" "cython==0.29.37" legacy-cgi
/content/bzenv/bin/buildozer --version


## 3) Projekt hochladen

ZIP auswählen, wenn kein Pfad in `ZIP` vorgegeben ist. Das Skript sucht automatisch den
Ordner mit `main.py` + `buildozer.spec`, prüft die Pfade im ZIP und entpackt nach
`/content/ps5pro`. Ein bereits vorhandenes `/content/ps5pro` wird dabei komplett ersetzt —
auch ein alter `.buildozer`-Ordner.

In [ ]:
#@title Projekt-ZIP einlesen {display-mode: 'form'}
import os, glob, shutil, zipfile

PROJ = '/content/ps5pro'
ZIP = '/content/ps5pro.zip'   # alternativ z.B. '/content/drive/MyDrive/ps5pro.zip'

def safe_extract(zip_path, work):
    shutil.rmtree(work, ignore_errors=True)
    os.makedirs(work, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        bad = [n for n in z.namelist() if n.startswith('/') or '..' in n.split('/')]
        assert not bad, f'unsichere Pfade im ZIP: {bad[:3]}'
        z.extractall(work)

def install_project(zip_path, dest=PROJ):
    work = '/content/_unzip'
    safe_extract(zip_path, work)
    root = work
    for _ in range(4):
        if os.path.exists(os.path.join(root, 'main.py')):
            break
        subs = [d for d in glob.glob(os.path.join(root, '*')) if os.path.isdir(d)]
        if len(subs) != 1:
            break
        root = subs[0]
    assert os.path.exists(os.path.join(root, 'main.py')), 'main.py nicht im ZIP gefunden'
    assert os.path.exists(os.path.join(root, 'buildozer.spec')), 'buildozer.spec nicht im ZIP gefunden'
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(root, dest)
    return dest

if not os.path.exists(ZIP):
    from google.colab import files
    up = files.upload()
    ZIP = os.path.join('/content', next(iter(up)))

print('Projekt entpackt nach:', install_project(ZIP))
print('Dateien:', sorted(os.listdir(PROJ)))

## 4) Optional: Build-Cache in Google Drive

Ein kompletter Build dauert lange, weil Python, Kivy und alle Abhängigkeiten für Android
kompiliert werden. `USE_CACHE = True` legt `.buildozer` in Google Drive ab und stellt es
beim nächsten Lauf wieder her — dann ist ein Rebuild meist in wenigen Minuten fertig.
Den Schalter nur umlegen, wenn du Drive nutzen willst (dann kommt eine Freigabe-Abfrage).

**Erster Lauf nach einem Toolchain-Wechsel:** `False` lassen. Ein alter Cache bringt sonst
das vorherige NDK und halbfertige Build-Verzeichnisse zurück.

In [ ]:
#@title Cache ein-/ausschalten {display-mode: 'form'}
import os, subprocess

USE_CACHE = False
CACHE_DIR = '/content/drive/MyDrive/ps5pro_cache'

def restore_cache():
    if not USE_CACHE:
        print('Cache deaktiviert - der erste Build dauert laenger.')
        return
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(CACHE_DIR, exist_ok=True)
    tar = os.path.join(CACHE_DIR, 'buildozer_cache.tar')
    if os.path.exists(tar):
        subprocess.run(['tar', '-xf', tar, '-C', PROJ], check=True)
        print('Cache wiederhergestellt:', tar)
    else:
        print('Noch kein Cache vorhanden - erster Build dauert laenger.')

restore_cache()

## 5) APK bauen

Hier läuft `buildozer -v android debug`. Das SDK/NDK wird von Buildozer automatisch
geladen (ca. 2 GB, landet in `/root/.buildozer`), Lizenzen werden über
`android.accept_sdk_license = True` in der `buildozer.spec` automatisch akzeptiert.
Gebaut wird für API 34 / NDK 28c und beide ARM-Architekturen.

Alle Ausgaben werden zusätzlich in `/content/buildozer.log` geschrieben, der Exit-Code
wird am Ende angezeigt. Bei einem Fehler springt die Zelle automatisch zur letzten
`Command failed`-Stelle im Log — das ist die Zeile mit der eigentlichen Ursache.

Wenn Colab die Sitzung währenddessen trennt: neu verbinden, Zelle 1–3 überspringen und
ab der Cache-Zelle weiterlaufen lassen.

In [ ]:
#@title buildozer android debug ausfuehren {display-mode: 'form'}
import glob, os, subprocess, time

PROJ = globals().get('PROJ', '/content/ps5pro')
BUILDOZER = '/content/bzenv/bin/buildozer'
LOG = '/content/buildozer.log'

assert os.path.isdir(PROJ), 'Projektordner fehlt - bitte Zelle 3 (Projekt-ZIP einlesen) ausfuehren'
assert os.path.exists(BUILDOZER), 'buildozer fehlt - bitte Zelle 2 (Buildozer installieren) ausfuehren'

env = dict(os.environ)
# Buildozer ruft intern `pip install --user` auf solange VIRTUAL_ENV fehlt - in einem
# venv verweigert pip das. Ohne diese zwei Zeilen bricht schon der Toolchain-Import ab.
env['VIRTUAL_ENV'] = '/content/bzenv'
env['PATH'] = '/content/bzenv/bin:' + env.get('PATH', '')
for cand in sorted(glob.glob('/usr/lib/jvm/java-17*') + glob.glob('/usr/lib/jvm/java-21*')):
    if os.path.isdir(cand):
        env['JAVA_HOME'] = cand
        break
print('JAVA_HOME:', env.get('JAVA_HOME', '(System-Java)'))

# sdkmanager fragt interaktiv nach den Lizenzen; in Colab bleibt das minutenlang stehen.
# Mit hinterlegten Hashes im SDK-Ordner wird gar nicht erst gefragt.
# WICHTIG: das SDK-Verzeichnis NIE vorab anlegen. buildozer prueft nur dessen Existenz
# und ueberspringt dann die Installation; danach fehlt tools/bin/sdkmanager und der
# Build bricht mit "sdkmanager is not installed" ab.
sdk = '/root/.buildozer/android/platform/android-sdk'
lic = os.path.join(sdk, 'licenses')
if os.path.isfile(os.path.join(sdk, 'tools', 'bin', 'sdkmanager')):
    os.makedirs(lic, exist_ok=True)
    for name, hashes in {
        'android-sdk-license': ['8933bad161af4178b1185d1a37fbf41ea5269c55',
                                'd56f5187479451eabf01fb78af6dfcb131a6481e',
                                '24333f8a63b6825ea9c5514f83c2829b004d1fee'],
        'android-sdk-preview-license': ['84831b9409646a918e30573bab4c9c91346d8abd'],
        'android-googletv-license': ['601085b94cd77f0b54ff86406957099ebe79c4d6'],
    }.items():
        with open(os.path.join(lic, name), 'w') as fh:
            fh.write(''.join(h + chr(10) for h in hashes))
    print('SDK-Lizenzen hinterlegt:', lic)
else:
    print('SDK noch nicht vorhanden - buildozer legt es an und beantwortet den Prompt selbst')

t0 = time.time()
proc = subprocess.Popen([BUILDOZER, '-v', 'android', 'debug'], cwd=PROJ, env=env,
                        stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
with open(LOG, 'w', encoding='utf-8') as lf:
    for line in proc.stdout:
        print(line, end='')
        lf.write(line)
rc = proc.wait()
print()
print(f'Exit-Code: {rc} ({time.time()-t0:.0f} s)')

if rc == 0:
    apks = sorted(glob.glob(os.path.join(PROJ, 'bin', '*.apk')), key=os.path.getmtime)
    print('APK:', apks[-1] if apks else 'keine APK in bin/ gefunden')
else:
    lines = open(LOG, encoding='utf-8', errors='replace').read().splitlines()
    idx = [i for i, l in enumerate(lines) if 'Command failed' in l]
    start = max(0, idx[-1] - 40) if idx else max(0, len(lines) - 60)
    print('--- Fehlerstelle aus dem Log ---')
    for l in lines[start:start + 60]:
        print(l)

## 6) APK herunterladen

Die fertige Datei liegt in `bin/`. Beim Klick auf „Herunterladen" im Browser-Popup
speicherst du sie auf dem Rechner und kopierst sie danach aufs Handy.

In [ ]:
#@title APK holen {display-mode: 'form'}
import glob, os, shutil, subprocess

apks = sorted(glob.glob(os.path.join(PROJ, 'bin', '*.apk')), key=os.path.getmtime)
assert apks, 'keine APK in bin/ gefunden - bitte das Build-Log pruefen'
apk = apks[-1]
print(apk, round(os.path.getsize(apk) / 1024 / 1024, 1), 'MB')

CACHE_DIR = globals().get('CACHE_DIR', '/content/drive/MyDrive/ps5pro_cache')
if globals().get('USE_CACHE') and os.path.isdir('/content/drive'):
    shutil.copy(apk, CACHE_DIR)
    if os.path.isdir(os.path.join(PROJ, '.buildozer')):
        subprocess.run(['tar', '-cf', os.path.join(CACHE_DIR, 'buildozer_cache.tar'), '-C', PROJ, '.buildozer'], check=False)
    print('APK und Cache in Drive gesichert:', CACHE_DIR)

from google.colab import files
files.download(apk)

In [ ]:
#@title Diagnose: Umgebung und letzter Fehler {display-mode: 'form'}
import glob, os, subprocess

def run(cmd, n=3):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
        out = (p.stdout + p.stderr).strip().splitlines()
        return out[:n] if out else ['(keine Ausgabe)']
    except Exception as e:
        return ['Fehler: %s' % e]

print('=== System ===')
print('Python:', run('python3 --version'))
print('Java:', run('java -version 2>&1'))
print('Platz:', run('df -h /content | tail -1'))
print('RAM:', run('free -h | head -2', 2))

print()
print('=== venv /content/bzenv ===')
print('vorhanden:', os.path.isdir('/content/bzenv'))
print('buildozer:', run('/content/bzenv/bin/buildozer --version'))
print('pip-Pakete:')
for l in run('/content/bzenv/bin/python -m pip list 2>/dev/null | grep -iE "buildozer|cython|python-for-android|virtualenv"', 8):
    print('  ', l)

print()
print('=== Projekt ===')
print('PROJ vorhanden:', os.path.isdir('/content/ps5pro'))
for f in ('main.py', 'buildozer.spec', 'icon.png', 'assets/icon_fg.png'):
    print('  ', f, os.path.exists('/content/ps5pro/' + f))

print()
print('=== SDK / NDK ===')
print('NDK:', sorted(glob.glob('/root/.buildozer/android/platform/android-ndk-*')) or 'keiner')
print('SDK:', sorted(glob.glob('/root/.buildozer/android/platform/android-sdk*')) or 'keiner')

print()
print('=== Letzter Fehler im Log ===')
log = '/content/buildozer.log'
if not os.path.exists(log):
    print('kein buildozer.log vorhanden - Zelle 5 lief noch nicht')
else:
    lines = open(log, encoding='utf-8', errors='replace').read().splitlines()
    print('Logzeilen:', len(lines))
    idx = [i for i, l in enumerate(lines) if 'Command failed' in l]
    start = max(0, idx[-1] - 25) if idx else max(0, len(lines) - 40)
    for l in lines[start:start + 45]:
        print(l)


## Fehlerbehebung

| Meldung | Ursache und Lösung |
| --- | --- |
| `Buildozer failed to execute the last command` | Immer die letzten Zeilen in `/content/buildozer.log` lesen — dort steht der echte Fehler. |
| Fehler zu `cython` / `Cython.Compiler` | `cython==0.29.37` in Zelle 2 installieren, nicht Cython 3. |
| Fehler zu `javac`, `sdkmanager` oder `AGP` | Java 17 installieren und `JAVA_HOME` prüfen (`/usr/lib/jvm/java-17-openjdk-amd64`). |
| `distutils` / `pkg_resources` fehlt | Zelle 2 nutzt bewusst die Python-3.11.9-Umgebung in `/content/bzenv`. |
| `zipfile.BadZipFile` oder `main.py nicht im ZIP gefunden` | ZIP neu packen; `main.py`, `buildozer.spec`, `icon.png` müssen enthalten sein. |
| Build bricht ohne Speicherfehler ab | `WORKER` in `main.py` (Standard 6) reduzieren, z. B. auf 4 — betrifft nur die Anzahl paralleler Scan-Threads. |
| `SDK license not accepted` | In der `buildozer.spec` steht `android.accept_sdk_license = True`; nach Änderung einmal `buildozer android clean` ausführen. |
| Fehler zu NDK/API nach einem Versionswechsel | `buildozer android clean`, `.buildozer` löschen und den Drive-Cache verwerfen. |
| Sitzung getrennt / 12-Stunden-Limit | Cache-Zelle nutzen, danach nur Zelle 5 und 6 erneut ausführen. |

## Nach dem Build

**Installieren:** APK auf das Handy kopieren, in den Einstellungen „Installation aus
unbekannten Quellen" für die Datei-App erlauben, APK antippen. Die Debug-APK ist mit
einem Debug-Schlüssel signiert und damit direkt installierbar (nicht für den Play Store).

**Laufzeit:** `minSdk 24` (Android 7.0) bis `targetSdk 34`, gebaut für `arm64-v8a`
(alle aktuellen Geräte; 32-Bit lässt sich in der Spec wieder ergänzen). Beim ersten Start fragt
Android nach Netzwerk-, Vibrations- und Benachrichtigungs-Rechten.

**Responsivität:** Das Layout rechnet durchgehend mit `dp`/`sp`, die Höhen von Titelleiste,
Ergebnis-Panel und Steuerleiste passen sich der Bildschirmhöhe an (`_apply_sizes` in
`main.py`), die Trefferkarten laufen in einer horizontal scrollbaren Liste. `orientation =
portrait` sperrt das Hochformat — so bleibt die Anordnung auf allen Größen stabil.

**Noch zu prüfen (nicht in Colab verifizierbar):** Der Build wurde hier nicht ausgeführt
(diese Umgebung hat kein Python und kein Android-SDK). Nach dem ersten erfolgreichen Lauf
bitte auf dem Gerät testen: Start ohne Absturz, Scan-Durchlauf, Händler-Auswahl, Alarm
(Benachrichtigung) und Drehen/Wechsel zwischen Geräten mit unterschiedlicher Auflösung.

**Play Store (optional):** Dafür braucht es `android.release_artifact = aab`, einen eigenen
Keystore und eine signierte Release-Version — das ist ein separater Schritt und nicht Teil
dieses Notebooks.